# Prerequisites: Creating Sample Agents

## Overview

Let's first start by creating agents to be evaluated. This tutorial creates two sample agents for evaluation using different frameworks:
- [Strands Agents SDK](https://strandsagents.com/)
- [LangGraph](https://www.langchain.com/langgraph)

Both agents uses Anthropic Claude Haiku 4.5 from Amazon Bedrock as the LLM model but you can use any model of your preference and they have identical capabilities:
- **Math Tool**: Tool to perform basic math operations
- **Weather Tool**: Dummy implementation for weather tool


The architecture looks as following:

![Architecture](../images/agent_architecture.png)

## Prerequisites
- Python 3.10+
- AWS credentials

In [ ]:
!pip install -r ../requirements.txt

## Setup

Import required packages and configure AWS session:

In [ ]:
import boto3
import json
import subprocess
import shutil
import time
import uuid
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print(f"Using region: {region}")

## Deploy Strands Agent
Let's deploy our Strands agent to AgentCore Runtime using the AgentCore CLI.

### CLI commands to deploy

```bash
# Install the AgentCore CLI (pin to 0.11.0)
npm install -g @aws/agentcore@0.11.0

# Create the project scaffold (name must be alphanumeric, max 23 chars)
agentcore create --name acevalstrands2 --framework Strands --model-provider Bedrock --defaults

# Copy the agent implementation
cp eval_agent_strands.py acevalstrands2/app/acevalstrands2/main.py

# Deploy to AgentCore Runtime
cd acevalstrands2 && agentcore deploy
```

The cells below execute these steps programmatically from the notebook.

In [ ]:
import os

# Step 1: Create AgentCore project scaffold for Strands agent
# Note: CLI name must be alphanumeric only (no underscores/hyphens), max 23 chars
STRANDS_PROJECT = "acevalstrands2"
if os.path.isdir(STRANDS_PROJECT):
    print(f"Project '{STRANDS_PROJECT}' already exists — skipping create.")
else:
    print(f"Creating Strands agent project '{STRANDS_PROJECT}' with AgentCore CLI...")
    subprocess.run(
        ["agentcore", "create", "--name", STRANDS_PROJECT,
         "--framework", "Strands", "--model-provider", "Bedrock", "--defaults"],
        check=True
    )

# Step 2: Copy the agent implementation into the project
shutil.copy("eval_agent_strands.py", f"{STRANDS_PROJECT}/app/{STRANDS_PROJECT}/main.py")
print(f"Agent code copied to {STRANDS_PROJECT}/app/{STRANDS_PROJECT}/main.py")

# Step 3: Add strands-agents-tools dependency (needed for the calculator tool)
pyproject_path = f"{STRANDS_PROJECT}/app/{STRANDS_PROJECT}/pyproject.toml"
with open(pyproject_path) as f:
    content = f.read()
if "strands-agents-tools" not in content:
    content = content.replace(
        '"bedrock-agentcore >= 1.0.3"',
        '"bedrock-agentcore >= 1.0.3",\n    "strands-agents-tools"'
    )
    with open(pyproject_path, "w") as f:
        f.write(content)
    print("Added strands-agents-tools to pyproject.toml")

# Step 3b: Populate aws-targets.json (agentcore create --defaults leaves it empty;
#           agentcore deploy -y requires a "default" target entry with account + region)
account_id = boto3.client("sts", region_name=region).get_caller_identity()["Account"]
targets_path = f"{STRANDS_PROJECT}/agentcore/aws-targets.json"
with open(targets_path, "w") as f:
    json.dump([{"name": "default", "description": f"Default target ({region})",
                "account": account_id, "region": region}], f, indent=2)
print(f"Wrote aws-targets.json (account={account_id}, region={region})")

# Step 4: Deploy to AgentCore Runtime (-y for non-interactive mode)
print(f"\nDeploying Strands agent (~5 minutes on first run)...")
subprocess.run(["agentcore", "deploy", "-y"], cwd=STRANDS_PROJECT, check=True)
print("Strands agent deployed.")

# Step 5: Look up agent ID and ARN via boto3 (paginate to handle large runtime lists)
cp = boto3.client("bedrock-agentcore-control", region_name=region)
paginator = cp.get_paginator("list_agent_runtimes")
strands_runtime = None
for page in paginator.paginate():
    for rt in page.get("agentRuntimeSummaries", page.get("agentRuntimes", [])):
        if STRANDS_PROJECT in rt.get("agentRuntimeId", ""):
            strands_runtime = rt
            break
    if strands_runtime:
        break

if strands_runtime is None:
    raise RuntimeError(f"Strands agent runtime not found (looking for '{STRANDS_PROJECT}'). Check deployment output above.")

agent_id_strands = strands_runtime["agentRuntimeId"]
agent_arn_strands = strands_runtime["agentRuntimeArn"]
print(f"\nStrands agent ready.")
print(f"  agent_id  : {agent_id_strands}")
print(f"  agent_arn : {agent_arn_strands}")

### Check status for deployment on AgentCore Runtime
Wait for deployment to be in ACTIVE status..

In [ ]:
def wait_for_agent(agent_id, name, region, max_wait=600, poll_interval=15):
    """Poll the AgentCore Runtime until the agent reaches a terminal state."""
    cp_client = boto3.client("bedrock-agentcore-control", region_name=region)
    end_statuses = ["READY", "FAILED", "CREATE_FAILED", "UPDATE_FAILED", "DELETE_FAILED"]
    elapsed = 0
    while elapsed < max_wait:
        resp = cp_client.get_agent_runtime(agentRuntimeId=agent_id)
        # API returns "status" or "agentRuntimeStatus" depending on SDK version
        status = resp.get("agentRuntimeStatus") or resp.get("status", "UNKNOWN")
        print(f"{name} status: {status}")
        if status in end_statuses:
            print(f"{name} deployment completed with status: {status}")
            return status
        time.sleep(poll_interval)
        elapsed += poll_interval
    raise TimeoutError(f"{name} did not reach a terminal state within {max_wait}s")

strands_status = wait_for_agent(agent_id_strands, "Strands", region)

### Invoke the Strands Agent on Runtime
Let's test the Strands agent by invoking the AgentCore Runtime endpoint.

**CLI equivalent:**
```bash
# From inside the project directory:
agentcore invoke "How much is 2+2?" --stream
```

The cells below invoke the agent programmatically using boto3.

In [ ]:
session_id_strands = str(uuid.uuid4())
print(f"Session ID: {session_id_strands}")

In [ ]:
agentcore_dp = boto3.client("bedrock-agentcore", region_name=region)

def invoke_agent(agent_arn, prompt, session_id):
    """Invoke an AgentCore Runtime agent and return its text response."""
    response = agentcore_dp.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        qualifier="DEFAULT",
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}).encode("utf-8"),
    )
    raw = response["response"].read().decode("utf-8")
    parts = []
    for line in raw.splitlines():
        if line.startswith("data: "):
            chunk = line[len("data: "):]
            try:
                chunk = json.loads(chunk)
            except Exception:
                pass
            parts.append(str(chunk))
    return "".join(parts) if parts else raw

result = invoke_agent(agent_arn_strands, "How much is 2+2?", session_id_strands)
print(result)

In [ ]:
result = invoke_agent(agent_arn_strands, "How is the weather now?", session_id_strands)
print(result)

In [ ]:
result = invoke_agent(agent_arn_strands, "Can you tell me the capital of the US?", session_id_strands)
print(result)

## Deploy LangGraph Agent to AgentCore Runtime

Let's also deploy our LangGraph agent to AgentCore Runtime using the CLI.

### CLI commands to deploy

```bash
# Name must be alphanumeric only (no underscores/hyphens), max 23 chars
agentcore create --name acevallanggraph2 --framework LangChain_LangGraph --model-provider Bedrock --defaults

# Copy the agent implementation
cp eval_agent_langgraph.py acevallanggraph2/app/acevallanggraph2/main.py

# Deploy
cd acevallanggraph2 && agentcore deploy
```

In [ ]:
# Step 1: Create AgentCore project scaffold for LangGraph agent
LANGGRAPH_PROJECT = "acevallanggraph2"
if os.path.isdir(LANGGRAPH_PROJECT):
    print(f"Project '{LANGGRAPH_PROJECT}' already exists — skipping create.")
else:
    print(f"Creating LangGraph agent project '{LANGGRAPH_PROJECT}' with AgentCore CLI...")
    subprocess.run(
        ["agentcore", "create", "--name", LANGGRAPH_PROJECT,
         "--framework", "LangChain_LangGraph", "--model-provider", "Bedrock", "--defaults"],
        check=True
    )

# Step 2: Copy the agent implementation
shutil.copy("eval_agent_langgraph.py", f"{LANGGRAPH_PROJECT}/app/{LANGGRAPH_PROJECT}/main.py")
print(f"Agent code copied to {LANGGRAPH_PROJECT}/app/{LANGGRAPH_PROJECT}/main.py")

# Step 3: Add langchain-community dependency (used by eval_agent_langgraph.py)
pyproject_path = f"{LANGGRAPH_PROJECT}/app/{LANGGRAPH_PROJECT}/pyproject.toml"
with open(pyproject_path) as f:
    content = f.read()
if "langchain-community" not in content:
    content = content.replace(
        '"bedrock-agentcore >= 1.0.3"',
        '"bedrock-agentcore >= 1.0.3",\n    "langchain-community"'
    )
    with open(pyproject_path, "w") as f:
        f.write(content)
    print("Added langchain-community to pyproject.toml")

# Step 3b: Populate aws-targets.json (required for non-interactive deploy)
targets_path = f"{LANGGRAPH_PROJECT}/agentcore/aws-targets.json"
with open(targets_path, "w") as f:
    json.dump([{"name": "default", "description": f"Default target ({region})",
                "account": account_id, "region": region}], f, indent=2)
print(f"Wrote aws-targets.json (account={account_id}, region={region})")

# Step 4: Deploy to AgentCore Runtime (-y for non-interactive mode)
print(f"\nDeploying LangGraph agent (~5 minutes on first run)...")
subprocess.run(["agentcore", "deploy", "-y"], cwd=LANGGRAPH_PROJECT, check=True)
print("LangGraph agent deployed.")

# Step 5: Look up agent ID and ARN via boto3 (paginate to handle large runtime lists)
paginator = cp.get_paginator("list_agent_runtimes")
langgraph_runtime = None
for page in paginator.paginate():
    for rt in page.get("agentRuntimeSummaries", page.get("agentRuntimes", [])):
        if LANGGRAPH_PROJECT in rt.get("agentRuntimeId", ""):
            langgraph_runtime = rt
            break
    if langgraph_runtime:
        break

if langgraph_runtime is None:
    raise RuntimeError(f"LangGraph agent runtime not found (looking for '{LANGGRAPH_PROJECT}'). Check deployment output above.")

agent_id_langgraph = langgraph_runtime["agentRuntimeId"]
agent_arn_langgraph = langgraph_runtime["agentRuntimeArn"]
print(f"\nLangGraph agent ready.")
print(f"  agent_id  : {agent_id_langgraph}")
print(f"  agent_arn : {agent_arn_langgraph}")

### Check the status of LangGraph agent
Now that we've deployed the LangGraph agent to AgentCore Runtime, let's check for it's deployment status

In [ ]:
langgraph_status = wait_for_agent(agent_id_langgraph, "LangGraph", region)

### Invoke the Langgraph Agent on Runtime
Test the LangGraph agent endpoint on AgentCore Runtime with a payload:

In [ ]:
session_id_langgraph = str(uuid.uuid4())
print(f"Session ID: {session_id_langgraph}")

In [ ]:
result = invoke_agent(agent_arn_langgraph, "What is 2+2?", session_id_langgraph)
print(result)

In [ ]:
result = invoke_agent(agent_arn_langgraph, "What is the weather now?", session_id_langgraph)
print(result)

In [ ]:
result = invoke_agent(agent_arn_langgraph, "Can you tell me the capital of the US?", session_id_langgraph)
print(result)

In [ ]:
print(f"Strands:   agent_id={agent_id_strands}, agent_arn={agent_arn_strands}, session={session_id_strands}")
print(f"LangGraph: agent_id={agent_id_langgraph}, agent_arn={agent_arn_langgraph}, session={session_id_langgraph}")

In [ ]:
%store agent_id_strands
%store agent_arn_strands
%store session_id_strands
%store agent_id_langgraph
%store agent_arn_langgraph
%store session_id_langgraph

## Next Steps

Now that you have all the required pre-requisites, let's go through the individual evaluation tutorials:
Continue with the evaluation tutorials:
- [01-creating-custom-evaluators](../01-creating-custom-evaluators/): Create custom evaluators
- [02-running-evaluations](../02-running-evaluations/): Run on-demand and online evaluations
- [03-advanced](../03-advanced/): : Advanced techniques and dashboards